[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/surface_curvature/surface_curvature.ipynb)

# Curvature of Quadric Surfaces

This example computes how quadric surfaces bend: at every point, the principal curvatures, the most and the least the surface curves in any direction, and the lines of curvature, the curves that follow those directions across the surface. How a surface bends is measured by two quadratic forms on its tangent directions, the second fundamental form, how fast the surface falls away from its tangent plane, and the first, the ordinary metric; solved against each other they give the principal curvatures and directions. For a central quadric the lines of curvature are where it meets the quadrics confocal with it.

A quadric is a polarity, a map `Plane <- Point` taking each point to its polar plane, and the surface is where a point lies on its own polar plane, `point & surface(point) == 0`. Read on directions, the ideal points, the same form `Point & surface(Point)` is the surface's Hessian; restricted to the tangent plane on both sides, it is the second fundamental form up to the gradient length, and its eigenvalues against the metric on directions are the principal curvatures. The confocal family shifts the quadric's shape by multiples of the metric, so the members through a point are the eigenvalues of one form.

In the notation of differential geometry the principal curvatures read as the roots of $\det(\mathrm{II} - \kappa\, \mathrm{I}) = 0$, the eigenvalues of the shape operator $\mathrm{I}^{-1}\mathrm{II}$, and by Dupin's theorem the three surfaces of a triply orthogonal confocal system $\sum_i x_i^2 / (a_i^2 - \lambda) = 1$ meet along their lines of curvature.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext
from numga.algebras import PGA3D as ga
from examples.util.animation import save_figure
from examples.surfaces.surface_curvature import render
from examples.surfaces.surface_curvature.core import nearest_root, pair, view_rays

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
# Finite points carry homogeneous weight, and directions are ideal points of weight zero.
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+
Direction = Point.derive.degenerate                 # Antivector-
# A polarity: each point to its polar plane.
Quadric = ga.gatype((Plane, Point))

# The three coordinate planes x = 0, y = 0, z = 0, and the plane at infinity.
axes = mv(Plane.derive.nondegenerate, np.eye(3))    # [axes] Plane+
w = mv.w                                            # [] Plane+
print(Direction, Quadric, sep="\n")

## 1. Quadrics as maps from points to planes

`axes & Point` is the map that reads a point's coordinates; multiplied by the planes again and summed, the plane dyads build a quadric, `Plane <- Point`, that sends each point to its polar plane. A point lies on the surface when it lies on its own polar plane, `point & surface(point) == 0`. On a ray, that form is a quadratic in the distance along it, its coefficients the form evaluated on the ray's heading and origin, and each pixel's ray meets the surface at the smaller positive root. The ellipsoid is drawn on the left and the hyperboloid of one sheet on the right.

In homogeneous coordinates the quadric reads as a symmetric $4 \times 4$ matrix $Q$, the surface as $\mathbf x^\top Q\, \mathbf x = 0$, and a point's polar plane as $Q\mathbf x$.

In [ ]:
# The ellipsoid's semi-axes, longest to shortest, and the hyperboloid's, with one negative sign for a saddle.
semi_axes = np.array([3.0, 2.0, 1.2])
hyperboloid_axes = np.array([1.6, 1.0, 1.0])
hyperboloid_signs = np.array([1.0, 1.0, -1.0])
# Each surface's camera elevation and azimuth in degrees, half the screen width, and the vertical clip height.
views = (((25, -60), 3.4), ((18, -60), 3.6))
n_pixels = 360
heights = (np.inf, 2.0)

In [ ]:
def quadric(weights: np.ndarray) -> Quadric:
    """A central quadric with one weight per coordinate axis, as a sum of plane dyads."""
    return (axes * (axes & Point) * weights).sum() - w * (w & Point)                    # [] Plane+ <- Point-


# A negative weight on one axis turns the ellipsoid into a hyperboloid of one sheet.
ellipsoid = quadric(1 / semi_axes**2)                                                   # [] Plane+ <- Point-
hyperboloid = quadric(hyperboloid_signs / hyperboloid_axes**2)                          # [] Plane+ <- Point-


def hit(surface: Quadric, origins: Point, heading: Direction) -> tuple[Point, Scalar]:
    """Where each ray first meets the surface, and the discriminant, negative where it misses."""
    form = Point & surface(Point)                                                       # [] Scalar+ <- (Point-, Point-)
    # Evaluating the bilinear form on a ray gives the coefficients of a quadratic in the distance along it.
    a, b, c = form(heading, heading), form(heading, origins), form(origins, origins)    # [...] Scalar+, Scalar+, Scalar+
    discriminant = b * b - a * c                                                        # [...] Scalar+
    # Step from each ray origin along the heading to the nearer positive root.
    return (origins + heading * nearest_root(a, b, discriminant)).normalized(), discriminant


rays = [view_rays(*view, extent, n_pixels) for view, extent in views]
hits = [hit(surface, *ray) for surface, ray in zip((ellipsoid, hyperboloid), rays)]
scenes = [(points, discriminant, surface(points), heading)
          for surface, (points, discriminant), (_, heading) in zip((ellipsoid, hyperboloid), hits, rays)]
save_figure(render.draw_surfaces(scenes, heights), "surface_curvature_surfaces");

## 2. The second form against the first

On directions the same map is the surface's Hessian: `Point & surface(Point)`, read on ideal points, measures how the quadric's value changes to second order along a direction. The metric on directions is the inner product of their duals, the planes through the origin they are normal to. At a point, the tangent plane is its polar plane, and projecting onto it is a map `Direction <- Direction`. The Hessian restricted to the tangent plane on both sides, divided by the gradient's length, is the second fundamental form. Its eigenvalues against the metric are the principal curvatures and one zero eigenvalue along the normal. The colours are red where both curvatures bend the same way and blue at a saddle. Curvature is measured on directions because that is where PGA's Euclidean metric lives, independent of a point's weight.

In the notation of differential geometry the principal curvatures read as the roots of $\det(\mathrm{II} - \kappa\, \mathrm{I}) = 0$, the eigenvalues of the shape operator $\mathrm{I}^{-1}\mathrm{II}$, and the Gaussian curvature as $K = \det\mathrm{II} / \det\mathrm{I}$.

In [ ]:
# The curvature at which the colour map saturates.
curvature_scale = 1.2

In [ ]:
# A direction's dual is the plane through the origin it is normal to; their inner product is the Euclidean metric.
metric = Direction.dual() | Direction.dual()                                            # [] Scalar+ <- (Direction-, Direction-)


def principal(surface: Quadric, points: Point) -> Scalar:
    """The two principal curvatures at points on the surface, in order; convex surfaces curve positively."""
    tangent = surface(points)                                                           # [...] Plane+
    # The normal is the tangent plane's dual less its weight, and the projector removes the normal component.
    normal = tangent.dual().cast(Direction)                                             # [...] Direction-
    projector = Direction - normal * ((tangent & Direction) / (tangent & normal))       # [...] Direction- <- Direction-
    # Restrict the Hessian to the tangent plane on both sides and divide by the gradient length.
    second_form = -(Point & surface(Point))(projector, projector) / metric(normal, normal).square_root()  # [...] Scalar+ <- (Direction-, Direction-)
    return pair(second_form.eigvalsh(metric))                                           # [..., curvatures] Scalar+


curvatures = [principal(surface, points) for surface, (points, _) in zip((ellipsoid, hyperboloid), hits)]  # [pixels, pixels, curvatures] Scalar+, Scalar+
save_figure(render.draw_gaussian_curvature(scenes, curvatures, heights, curvature_scale), "surface_curvature_gaussian");

In [ ]:
tip_curvatures = principal(ellipsoid, mv.point([semi_axes[0], 0.0, 0.0]))               # [curvatures] Scalar+
print("curvatures at the tip of the long axis:", tip_curvatures.to_array(), "  long over each other semi-axis squared:", semi_axes[0] / semi_axes[1:]**2)

## 3. Lines of curvature from the confocal family

A central quadric sits in a family of confocal quadrics, the quadrics that share its foci, and its lines of curvature are where it meets them. In extensor form the family is a shift: the dual quadric read on directions is the quadric's shape, and each member shifts that shape by a multiple of the metric. A point lies on a member where the shape minus the point's own dyad is singular against the metric, so the members through a point are the eigenvalues of that one form. One eigenvalue is zero for the surface itself, and the level sets of the other two draw the net.

In Cartesian coordinates the confocal family of an ellipsoid reads as $\frac{x^2}{a^2 - \lambda} + \frac{y^2}{b^2 - \lambda} + \frac{z^2}{c^2 - \lambda} = 1$, and the three values of $\lambda$ through a point as its ellipsoidal coordinates.

In [ ]:
def confocal(surface: Quadric, points: Point) -> Scalar:
    """The parameters of the confocal quadrics through each point."""
    dual_surface = surface.inverse()                                                    # [] Point- <- Plane+
    # The pole of the plane at infinity is the centre, normalised to unit weight.
    pole = dual_surface(w)                                                              # [] Point-
    centre = pole / (w & pole)                                                          # [] Point-
    # Map each direction to the plane through the centre normal to it, and take the dual of its pole.
    central_planes = Direction.dual() - w * (Direction.dual() & centre)                 # [] Plane+ <- Direction-
    shape = dual_surface(central_planes).dual()                                         # [] Plane+ <- Direction-
    # Each point's offset from the centre, dualised into the plane normal to that displacement.
    offset_planes = (points - centre).dual()                                            # [...] Plane+
    # Subtract the point's own dyad from the shape; the eigenvalues are the confocal members through the point.
    confocal_form = Direction & (shape - offset_planes * (offset_planes & Direction))   # [...] Scalar+ <- (Direction-, Direction-)
    return pair(confocal_form.eigvalsh(metric))                                         # [..., members] Scalar+


parameters = [confocal(surface, points) for surface, (points, _) in zip((ellipsoid, hyperboloid), hits)]  # [pixels, pixels, members] Scalar+, Scalar+
save_figure(render.draw_curvature_lines(scenes, curvatures, parameters, heights, curvature_scale), "surface_curvature_lines");

The four points where the lines on the ellipsoid swirl together are its umbilics: there the surface curves equally in every direction, the two principal curvatures agree, and the two confocal parameters meet. They lie in the plane of the longest and shortest axes.

In [ ]:
# The umbilics lie in the plane of the longest and shortest semi-axes.
long_axis, middle_axis, short_axis = semi_axes
x_coord = long_axis * np.sqrt((long_axis**2 - middle_axis**2) / (long_axis**2 - short_axis**2))
z_coord = short_axis * np.sqrt((middle_axis**2 - short_axis**2) / (long_axis**2 - short_axis**2))
umbilics = mv.point([[x_coord, 0, z_coord], [-x_coord, 0, z_coord], [x_coord, 0, -z_coord], [-x_coord, 0, -z_coord]])  # [umbilics] Point-
umbilic_curvatures = principal(ellipsoid, umbilics)                                     # [umbilics, curvatures] Scalar+
umbilic_parameters = confocal(ellipsoid, umbilics)                                      # [umbilics, members] Scalar+
print("principal curvatures at the umbilics:\n", umbilic_curvatures.to_array())
print("confocal parameters there:\n", umbilic_parameters.to_array())

In [ ]:
# checks
np.testing.assert_allclose(tip_curvatures.kernel[:, 0], semi_axes[0] / semi_axes[1:]**2, rtol=1e-8)
# The two principal curvatures and the two confocal parameters agree at the umbilics.
np.testing.assert_allclose(umbilic_curvatures[:, 0].kernel, umbilic_curvatures[:, 1].kernel, rtol=1e-8)
np.testing.assert_allclose(umbilic_parameters[:, 0].kernel, umbilic_parameters[:, 1].kernel, rtol=1e-6)